Crea volumen, carpetas y tablas Bronze/Silver para el pronostico ECMWF (cf via TIGGE/cdsapi, fc via Open Data, pf via TIGGE/cdsapi - ensemble completo de 50 miembros perturbados). Bronze guarda todo el bounding box descargado; Silver aplica el recorte al poligono real de las 3 sub-cuencas.

In [ ]:
spark.sql('CREATE CATALOG IF NOT EXISTS weather')
spark.sql('CREATE SCHEMA IF NOT EXISTS weather.raw')
spark.sql('CREATE SCHEMA IF NOT EXISTS weather.bronze')
spark.sql('CREATE SCHEMA IF NOT EXISTS weather.silver')
spark.sql('CREATE VOLUME IF NOT EXISTS weather.raw.ecmwf_volume')

dbutils.fs.mkdirs('/Volumes/weather/raw/ecmwf_volume/fc_opendata/raw')
dbutils.fs.mkdirs('/Volumes/weather/raw/ecmwf_volume/fc_opendata/json')
dbutils.fs.mkdirs('/Volumes/weather/raw/ecmwf_volume/cf_tigge/raw')
dbutils.fs.mkdirs('/Volumes/weather/raw/ecmwf_volume/cf_tigge/json')
dbutils.fs.mkdirs('/Volumes/weather/raw/ecmwf_volume/pf_tigge/raw')
dbutils.fs.mkdirs('/Volumes/weather/raw/ecmwf_volume/pf_tigge/json')


## Tablas Bronze

Espejo fiel de lo descargado (todo el bounding box, sin recortar al poligono). `tp_mm` ya viene normalizado a milimetros en el aplanado (fc: metros x1000; cf: kg/m2, ya equivalente a mm, sin conversion).

In [ ]:
spark.sql('''
CREATE TABLE IF NOT EXISTS weather.bronze.ecmwf_forecast_fc (
  run_date DATE,
  run_time STRING,
  step_hours INT,
  valid_date DATE,
  valid_datetime TIMESTAMP,
  latitude DOUBLE,
  longitude DOUBLE,
  tp_mm DOUBLE,
  tipo STRING,
  source_api STRING,
  source_file STRING,
  extracted_at TIMESTAMP,
  ingestion_date DATE,
  loaded_at TIMESTAMP,
  updated_at TIMESTAMP
) USING DELTA
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS weather.bronze.ecmwf_forecast_cf (
  run_date DATE,
  run_time STRING,
  step_hours INT,
  valid_date DATE,
  valid_datetime TIMESTAMP,
  latitude DOUBLE,
  longitude DOUBLE,
  number INT,
  tp_mm DOUBLE,
  tipo STRING,
  source_api STRING,
  source_file STRING,
  extracted_at TIMESTAMP,
  ingestion_date DATE,
  loaded_at TIMESTAMP,
  updated_at TIMESTAMP
) USING DELTA
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS weather.bronze.ecmwf_forecast_pf (
  run_date DATE,
  run_time STRING,
  step_hours INT,
  valid_date DATE,
  valid_datetime TIMESTAMP,
  latitude DOUBLE,
  longitude DOUBLE,
  number INT,
  tp_mm DOUBLE,
  tipo STRING,
  source_api STRING,
  source_file STRING,
  extracted_at TIMESTAMP,
  ingestion_date DATE,
  loaded_at TIMESTAMP,
  updated_at TIMESTAMP
) USING DELTA
''')


## Tablas Silver

Solo los puntos de grilla dentro del buffer (~0.15 grados) del poligono union de las 3 sub-cuencas (`SIG/subcuencas_modelo.geojson`).

In [ ]:
spark.sql('''
CREATE TABLE IF NOT EXISTS weather.silver.ecmwf_forecast_fc_basin (
  run_date DATE,
  run_time STRING,
  step_hours INT,
  valid_date DATE,
  valid_datetime TIMESTAMP,
  latitude DOUBLE,
  longitude DOUBLE,
  tp_mm DOUBLE,
  subcuenca_id INT,
  subcuenca_nombre STRING,
  source_table STRING,
  processed_at TIMESTAMP,
  updated_at TIMESTAMP
) USING DELTA
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS weather.silver.ecmwf_forecast_cf_basin (
  run_date DATE,
  run_time STRING,
  step_hours INT,
  valid_date DATE,
  valid_datetime TIMESTAMP,
  latitude DOUBLE,
  longitude DOUBLE,
  number INT,
  tp_mm DOUBLE,
  subcuenca_id INT,
  subcuenca_nombre STRING,
  source_table STRING,
  processed_at TIMESTAMP,
  updated_at TIMESTAMP
) USING DELTA
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS weather.silver.ecmwf_forecast_pf_basin (
  run_date DATE,
  run_time STRING,
  step_hours INT,
  valid_date DATE,
  valid_datetime TIMESTAMP,
  latitude DOUBLE,
  longitude DOUBLE,
  number INT,
  tp_mm DOUBLE,
  subcuenca_id INT,
  subcuenca_nombre STRING,
  source_table STRING,
  processed_at TIMESTAMP,
  updated_at TIMESTAMP
) USING DELTA
''')


In [ ]:
for table_name in [
    'weather.bronze.ecmwf_forecast_fc', 'weather.bronze.ecmwf_forecast_cf', 'weather.bronze.ecmwf_forecast_pf',
    'weather.silver.ecmwf_forecast_fc_basin', 'weather.silver.ecmwf_forecast_cf_basin', 'weather.silver.ecmwf_forecast_pf_basin',
]:
    print(f'DESCRIBE {table_name}')
    spark.sql(f'DESCRIBE {table_name}').show(truncate=False)
